In [1]:
import numpy as np
import matplotlib.pyplot as plt
from Utilities import extractor
import uproot
import awkward as ak    

x_MH200=extractor("Dati/Tprime_tAq_1800_MH200_LH_2017.root", "Events")


file=uproot.open("Dati/Tprime_tAq_1800_MH200_LH_2017.root")
tree=file["Events"]
booleans= tree.arrays(["FatJet_isMatchedWithA"], library="ak")
Fatjet_isMatchedWithA= booleans["FatJet_isMatchedWithA"]

#Filtriamo i dati

mask = ak.flatten(Fatjet_isMatchedWithA) == 1
x_filtered = x_MH200[mask]


/home/riccardo/anaconda3/envs/rootnev/lib/python3.14/site-packages/cppyy/__init__.py:374: UserWarning: CPyCppyy API not found (tried: /home/riccardo/anaconda3/envs/rootnev/include/site/python3.14); set CPPYY_API_PATH envar to the 'CPyCppyy' API directory to fix
  warnings.warn("CPyCppyy API not found (tried: %s); "
/home/riccardo/anaconda3/envs/rootnev/lib/python3.14/site-packages/awkward/_nplikes/array_module.py:289: RuntimeWarning: invalid value encountered in divide
  return impl(*broadcasted_args, **(kwargs or {}))


In [2]:
from scipy.special import voigt_profile
from iminuit import Minuit
from iminuit.cost import LeastSquares

x_plot=list(x_filtered)
x_plot.sort()

def voigt(x, norm, mu, sigma, gamma):
    return voigt_profile(x-mu, sigma, gamma) * norm

bin_counts, bin_edges = np.histogram(x_plot, bins=50)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
bin_width = bin_edges[1] - bin_edges[0]
bin_densities = bin_counts / (len(x_plot) * bin_width)  # Densità normalizzata
yerr=np.sqrt(bin_counts) / (len(x_plot) * bin_width) # Errore standard per i dati binned

ls_voigt=LeastSquares(bin_centers, bin_densities, yerr, model=voigt)

m_voigt=Minuit(ls_voigt,  norm=1, mu=200, sigma=5, gamma=1)
m_voigt.limits["mu"]= (175, 225)
m_voigt.limits["sigma"]= (0.1, 20)
m_voigt.limits["gamma"]= (0.01, 10)
m_voigt.migrad()


┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 1.446e+04 (χ²/ndof = 314.3)│              Nfcn = 167              │
│ EDM = 3e-05 (Goal: 0.0002)       │            time = 0.3 sec            │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬───────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name  │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼───────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ norm  │   0.780   │   0.004   │            │            │         │         │       │
│ 1 │ mu    │  200.28   │   0.11    │            │            │   175   │   225   │       │
│ 2 │ sigma │   10.93   │   0.19    │            │            │   0.1   │   20    │       │
│ 3 │ gamma │   7.55    │   0.17    │            │            │  0.01   │   10    │       │
└───┴───────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌───────┬─────────────────────────────────────────┐
│       │      norm        mu     sigma     gamma │
├───────┼─────────────────────────────────────────┤
│  norm │  1.62e-05 -0.013e-3 -0.149e-3  0.175e-3 │
│    mu │ -0.013e-3    0.0122    -0.004    -0.003 │
│ sigma │ -0.149e-3    -0.004    0.0347    -0.026 │
│ gamma │  0.175e-3    -0.003    -0.026    0.0298 │
└───────┴─────────────────────────────────────────┘

In [4]:
fit_MH200_values={}
fit_MH200_errors={}

fit_values={'MH200': fit_MH200_values,}
fit_errors={'MH200_errors': fit_MH200_errors}

for param in m_voigt.parameters:
    fit_MH200_values[param] = m_voigt.values[param]

for error in m_voigt.parameters:    #Qui non ho capito come fa a capire che deveestarre gli errori 
    fit_MH200_errors[error] = m_voigt.errors[error]

print(fit_MH200_values)
print(fit_MH200_errors)

import json
#QUi sono andato di metodo oragutang, ho deciso di voler fare 2 file separati peer errori e valori 
#Ho tenuto lo stesso quello con tutti i valori, casomai cambiassi idea

with open("fit_results.json", "r") as f:
    results=json.load(f)

with open("fit_values.json", "r") as g:
    values=json.load(g) 

with open("fit_errors.json", "r") as h:
    errors=json.load(h)


results["MH200"]=fit_MH200_values
results["MH200_errors"]=fit_MH200_errors

with open("fit_results.json", "w") as f:
    json.dump(results, f, indent=1)

values["MH200"]=fit_MH200_values
with open("fit_values.json", "w") as g:
    json.dump(values, g, indent=1)  

errors["MH200_errors"]=fit_MH200_errors
with open("fit_errors.json", "w") as h:
    json.dump(errors, h, indent=1)  


{'norm': 0.7796882337578827, 'mu': 200.2780170632862, 'sigma': 10.925645883569494, 'gamma': 7.5492906204296535}
{'norm': 0.004027377703588132, 'mu': 0.11034690228341049, 'sigma': 0.18635594911053044, 'gamma': 0.1725504251603498}
